# Build Corrected Adaptive Sequence Delta v2

This CPU notebook materializes corrected strict-prior train/validation sequence arrays after Notebook 28 passed exact parity. It stores only `example_id + recent/session state`, not duplicate user/item features.

Large output is built on Colab local disk and uploaded to the private Hugging Face model repository under:

```text
corrected_sequences/v2/
```

Drive receives only the small manifest and location file. Use a **High-RAM CPU** runtime. GPU is unnecessary. Add a Colab secret named `HF_WRITE_TOKEN` with write permission to `khoa1505/recsys-artifacts`; `HF_TOKEN` remains the read-token fallback.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
WORK_ROOT = Path('/content/recsys_corrected_sequence_build_v2')
SOURCE_ROOT = WORK_ROOT / 'runtime'
BUILD_WORK = WORK_ROOT / 'build_work'
LOCAL_REPO_ROOT = WORK_ROOT / 'hf_upload'
FULL_OUTPUT = LOCAL_REPO_ROOT / 'corrected_sequences/v2'
SMOKE_OUTPUT = WORK_ROOT / 'smoke_output'
SMOKE_WORK = WORK_ROOT / 'smoke_work'
SPARK_TMP = Path('/content/spark-corrected-build-v2')
DRIVE_REPORT_DIR = DRIVE_ROOT / 'artifacts/adaptive/corrected_sequences_v2'
for path in (SOURCE_ROOT, BUILD_WORK, LOCAL_REPO_ROOT, DRIVE_REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('output:', FULL_OUTPUT)
print('Drive report only:', DRIVE_REPORT_DIR)

In [ ]:
required = {
    'pyspark': 'pyspark==3.5.6',
    'pyarrow': 'pyarrow',
    'pandas': 'pandas',
    'huggingface_hub': 'huggingface_hub>=0.32',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
required_paths = [
    BASE_GOLD_ROOT / 'history/events',
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing inputs:\n' + '\n'.join(missing_paths)
free_gib = shutil.disk_usage('/content').free / 1024**3
assert free_gib >= 35, f'At least 35 GiB local disk is required; only {free_gib:.1f} GiB is free.'
print(f'preflight: PASS | local disk free: {free_gib:.1f} GiB')

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
RUNTIME_FILENAME = 'runtime/corrected_sequence_build_runtime_v2.zip'
RUNTIME_EXPECTED_SHA256 = '3f46ac263d674fb53e813bba9c98ff3ce63358bafd6bd67e10d184d1f4814e46'

def normalize_hf_token(raw):
    if isinstance(raw, bytes):
        raw = raw.decode('utf-8')
    if isinstance(raw, str):
        raw = raw.strip()
        return raw if raw.startswith('hf_') else None
    if isinstance(raw, dict):
        for value in raw.values():
            token = normalize_hf_token(value)
            if token:
                return token
    if isinstance(raw, (list, tuple)):
        for value in raw:
            token = normalize_hf_token(value)
            if token:
                return token
    return None

def colab_secret(name):
    try:
        return normalize_hf_token(userdata.get(name))
    except Exception:
        return None

HF_TOKEN = colab_secret('HF_TOKEN')
if not HF_TOKEN:
    HF_TOKEN = normalize_hf_token(getpass('Paste Hugging Face READ token: '))
HF_WRITE_TOKEN = colab_secret('HF_WRITE_TOKEN') or HF_TOKEN
assert HF_TOKEN and HF_WRITE_TOKEN, 'Valid Hugging Face tokens are required.'
runtime_zip = Path(hf_hub_download(
    repo_id=HF_REPO_ID, filename=RUNTIME_FILENAME, repo_type='model',
    token=HF_TOKEN, force_download=True,
))

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

assert sha256(runtime_zip) == RUNTIME_EXPECTED_SHA256, 'Runtime ZIP hash mismatch'
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None
    archive.extractall(SOURCE_ROOT)
BUILD_SCRIPT = SOURCE_ROOT / 'scripts/build_corrected_adaptive_sequence_delta.py'
assert BUILD_SCRIPT.exists()
compile(BUILD_SCRIPT.read_text(), str(BUILD_SCRIPT), 'exec')
print('runtime integrity: PASS')

In [ ]:
# Shared runner streams logs and preserves context checkpoints after failures.
def run_builder(output_dir, work_dir, extra_args, log_name):
    command = [
        sys.executable, str(BUILD_SCRIPT),
        '--adaptive-root', str(ADAPTIVE_ROOT),
        '--base-gold-root', str(BASE_GOLD_ROOT),
        '--output-dir', str(output_dir),
        '--work-dir', str(work_dir),
        '--max-sequence-len', '30',
        '--max-session-sequence-len', '10',
        '--overwrite',
        *extra_args,
    ]
    env = os.environ.copy()
    env.update({
        'SPARK_MASTER': 'local[2]',
        'PYSPARK_SUBMIT_ARGS': f'--driver-memory 8g --conf spark.local.dir={SPARK_TMP} pyspark-shell',
        'PYTHONPATH': str(SOURCE_ROOT) + os.pathsep + env.get('PYTHONPATH', ''),
    })
    log_path = Path('/content') / log_name
    process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with log_path.open('w') as log:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
    code = process.wait()
    if code != 0:
        tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-240:])
        raise RuntimeError(f'Builder failed with exit code {code}:\n{tail}')
    manifest_path = Path(output_dir) / 'manifest.json'
    assert manifest_path.exists(), f'Missing manifest: {manifest_path}'
    manifest = json.loads(manifest_path.read_text())
    assert manifest['quality']['passed'] is True
    return manifest

## Smoke build — 3 users

This validates the real Drive schemas, joins, ZSTD Parquet write, and quality gates before the full build.

In [ ]:
for path in (SMOKE_OUTPUT, SMOKE_WORK):
    shutil.rmtree(path, ignore_errors=True)
smoke_manifest = run_builder(
    SMOKE_OUTPUT, SMOKE_WORK,
    ['--max-users', '3', '--shuffle-partitions', '8', '--output-partitions', '4', '--rebuild-contexts'],
    'corrected_sequence_smoke.log',
)
print(json.dumps(smoke_manifest, indent=2))
shutil.rmtree(SMOKE_OUTPUT, ignore_errors=True)
shutil.rmtree(SMOKE_WORK, ignore_errors=True)
print('smoke build: PASS')

## Full train/validation build

Global and session contexts are checkpointed under `BUILD_WORK`. If a later stage fails while the Colab runtime remains alive, rerunning this cell reuses completed context checkpoints.

In [ ]:
FULL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
full_manifest = run_builder(
    FULL_OUTPUT, BUILD_WORK,
    ['--shuffle-partitions', '64', '--output-partitions', '96'],
    'corrected_sequence_full_build.log',
)
output_bytes = sum(path.stat().st_size for path in FULL_OUTPUT.rglob('*') if path.is_file())
print(json.dumps(full_manifest, indent=2))
print(f'local corrected delta size: {output_bytes / 1024**3:.2f} GiB')

## Upload large dataset to private Hugging Face

The large-folder uploader is resumable. If the connection drops, rerun this cell; already uploaded files are reused.

In [ ]:
api = HfApi(token=HF_WRITE_TOKEN)
api.repo_info(repo_id=HF_REPO_ID, repo_type='model', token=HF_WRITE_TOKEN)
if hasattr(api, 'upload_large_folder'):
    api.upload_large_folder(
        repo_id=HF_REPO_ID, repo_type='model', folder_path=str(LOCAL_REPO_ROOT),
        num_workers=4, print_report=True, print_report_every=60,
    )
else:
    api.upload_folder(
        repo_id=HF_REPO_ID, repo_type='model', folder_path=str(FULL_OUTPUT),
        path_in_repo='corrected_sequences/v2',
        commit_message='Upload corrected adaptive sequence delta v2',
    )
remote_manifest_path = Path(hf_hub_download(
    repo_id=HF_REPO_ID, filename='corrected_sequences/v2/manifest.json',
    repo_type='model', token=HF_TOKEN, force_download=True,
))
remote_manifest = json.loads(remote_manifest_path.read_text())
assert remote_manifest['version'] == 'corrected_adaptive_sequences_v2'
assert remote_manifest['quality']['passed'] is True
assert remote_manifest['quality']['rows'] == full_manifest['quality']['rows']
print('remote upload verification: PASS')

In [ ]:
# Save only tiny references on Drive, then release local build storage.
shutil.copy2(FULL_OUTPUT / 'manifest.json', DRIVE_REPORT_DIR / 'manifest.json')
location = {
    'repo_id': HF_REPO_ID,
    'repo_type': 'model',
    'path': 'corrected_sequences/v2',
    'version': 'corrected_adaptive_sequences_v2',
}
(DRIVE_REPORT_DIR / 'hf_location.json').write_text(json.dumps(location, indent=2))
print(json.dumps(location, indent=2))
print('Drive manifest:', DRIVE_REPORT_DIR / 'manifest.json')
shutil.rmtree(BUILD_WORK, ignore_errors=True)
shutil.rmtree(LOCAL_REPO_ROOT, ignore_errors=True)
shutil.rmtree(SPARK_TMP, ignore_errors=True)
print('local build storage released')